The profiling script performs deep inspection of Gold layer tables to generate a centralized audit trail. It calculates volumetric and quality metrics to ensure data reliability for further analytics to be done on these tables.

**Key Features**
- Dynamic Table Discovery: Iterates through a JSON-defined list of datasets provided via Databricks widgets.
- Automated Audit Schema: Dynamically creates the audit schema if it does not exist within the Silver catalog.
- Volumetric Analysis: Captures total row counts, column counts, and unique record counts to verify deduplication success.
- Data Quality KPIs: Calculates total null counts and null_percent across all columns to flag data gaps.
- Incremental Auditing: Uses Delta Lake append mode with mergeSchema to maintain a historical log of data profiles over time.

In [0]:
import json
import warnings
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# --- 1. CONFIGURATION ---
dbutils.widgets.text("datasets_json", '[]', "Datasets List (JSON Array)")

try:
    from schema_mapping import GOLD_CATALOG, GOLD_SCHEMA
except ImportError:
    GOLD_CATALOG, GOLD_SCHEMA = "data_gold", "gold"

AUDIT_SCHEMA = "audit"
PROFILE_TABLE = f"{GOLD_CATALOG}.{AUDIT_SCHEMA}.profile_summary"

# FIX: Added the missing tables to the map to remove "[INFO] No discovery metadata" warnings
ENTERPRISE_METADATA_MAP = {
    "dim_geography_gold": {
        "description": "Master Geography Dimension with hierarchical mapping of City, County, and State.",
        "columns": {"city": "Official city name."}
    },
    "dim_county_gold": {
        "description": "County dimension for regional aggregation.",
        "columns": {"county": "Name of the county."}
    },
    "dim_state_gold": {
        "description": "State dimension for high-level filtering.",
        "columns": {"state": "Full state name."}
    },
    "dim_region_type": {
        "description": "Lookup table for geographic grains (e.g., zip, city, county).",
        "columns": {"region_type": "The category of geographic grain."}
    },
    "dim_metric_dictionary": {
        "description": "Business Glossary mapping internal headers to human-readable definitions.",
        "columns": {"definition": "Official business logic definition."}
    },
    "fact_market_metrics_gold": {
        "description": "Enterprise Fact table for time-series market metrics.",
        "columns": {"date": "Measurement period."}
    }
}

def apply_enterprise_metadata(table_path, dataset_name):
    """
    Applies comments with 'Managed Table' detection to prevent DLT/Streaming errors.
    """
    metadata = ENTERPRISE_METADATA_MAP.get(dataset_name.lower())
    if not metadata:
        # This will now only trigger if a table is truly missing from the map above
        print(f"[INFO] No discovery metadata defined for {dataset_name}.")
        return

    try:
        # 1. Apply Table/View Comment
        spark.sql(f"COMMENT ON TABLE {table_path} IS '{metadata['description']}'")
        
        # 2. Apply Column Comments
        for col_name, comment in metadata['columns'].items():
            if col_name in spark.table(table_path).columns:
                spark.sql(f"COMMENT ON COLUMN {table_path}.{col_name} IS '{comment}'")
        
        print(f"[SUCCESS] Discovery metadata applied to {table_path}.")
    except Exception as e:
        # FIX: Catch DLT/Streaming table restriction and log it as a skip instead of an error
        if "STREAMING_TABLE_OPERATION_NOT_ALLOWED" in str(e):
            print(f"[SKIP METADATA] {dataset_name} is a Streaming/DLT table. Metadata must be updated in the DLT pipeline code.")
        else:
            print(f"[WARNING] Could not apply metadata to {table_path}: {e}")

def profile_gold_tables():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{AUDIT_SCHEMA}")
    
    try:
        datasets_raw = dbutils.widgets.get("datasets_json")
        dataset_list = json.loads(datasets_raw)
    except Exception as e:
        print(f"[ERROR] Invalid widget input: {e}")
        return

    if not dataset_list:
        return

    all_profiles = []

    for ds in dataset_list:
        table_path = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.{ds.lower()}"
        
        try:
            if not spark.catalog.tableExists(table_path):
                print(f"[SKIP] Table {table_path} not found.")
                continue
            
            # Apply metadata first
            apply_enterprise_metadata(table_path, ds)
            
            # Start Profiling
            print(f"[PROFILING] {ds}...")
            df = spark.table(table_path)
            row_count = df.count()
            col_count = len(df.columns)
            
            # Efficiency check: if empty, skip heavy null/unique counts
            if row_count == 0:
                print(f"[INFO] {ds} is empty. Skipping detailed metrics.")
                continue

            null_counts_expr = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
            null_data = df.select(null_counts_expr).collect()[0].asDict()
            total_nulls = sum(null_data.values())
            
            null_percent = (total_nulls / (row_count * col_count)) * 100
            unique_count = df.dropDuplicates().count() 
            
            all_profiles.append({
                "dataset_name": ds,
                "layer": "GOLD",
                "row_count": row_count,
                "column_count": col_count,
                "null_count": total_nulls,
                "null_percent": round(float(null_percent), 2),
                "unique_count": unique_count,
                "columns": ", ".join(df.columns),
                "profile_timestamp": datetime.now()
            })
            
        except Exception as e:
            print(f"[ERROR] Could not profile {ds}: {str(e)}")

    if all_profiles:
        profile_df = spark.createDataFrame(all_profiles)
        profile_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(PROFILE_TABLE)
        print("[SUCCESS] All profiling and metadata tasks completed.")

if __name__ == "__main__":
    profile_gold_tables()

In [0]:
# Arguments to be passed here from job and as task params in the job
#["dim_county","dim_state", "dim_metric_dictionary", "dim_region_type","dim_geography_gold","fact_market_metrics_gold"]

# Since MVs don't have data stored in explicitly, running profiling on them won't give correct numbers but it will provide correct ones for objects which have data stored in them like Streaming tables and normal tables